# Autoencoder for denoising task using U-NET architecture

**Attention**: Parts of code which are not explained here, are explained in `autoencoder.ipynb`

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
BATCH_SIZE = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LEARNING_RATE = 1e-3
EPOCHS = 10
NOISE_FACTOR = 0.25

In [ ]:
train_data = datasets.CIFAR10(root='./data', train=True, transform=transforms.ToTensor(), download=True)
test_data = datasets.CIFAR10(root='./data', train=False, transform=transforms.ToTensor(), download=True)

train_dataloader = DataLoader(dataset= train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
test_dataloader = DataLoader(dataset=test_data, batch_size=10, shuffle=False, pin_memory=True)

In [ ]:
train_data.class_to_idx

In [ ]:
train_data[0][0].shape

## Architectural Overview: U-Net Denoising Autoencoder

This section breaks down the implementation details, design decisions, and mechanics of the `UNET_Autoencoder` architecture.

---

## 1. High-Level Concept: Why U-Net over Standard Autoencoders?

Standard Convolutional Autoencoders compress an input image into a low-dimensional **bottleneck (latent space)** and then attempt to reconstruct the original image from that compressed representation. 

* **The Problem with Standard Autoencoders:** As spatial dimensions shrink ($32\times32 \to 16\times16 \to 8\times8 \to 4\times4 \to 2\times2$), fine-grained spatial details (like sharp edges, high-frequency textures, and exact pixel boundaries) are irreversibly lost. The decoder must "guess" these details, leading to **blurry reconstructions**.
* **The U-Net Solution:** U-Net introduces **Skip Connections**. It routes feature maps directly from intermediate layers of the **Encoder** to corresponding layers in the **Decoder**.
  * **Encoder** captures *what* is in the image (high-level semantic concepts).
  * **Skip Connections** preserve *where* things are (exact spatial location and sharp structural details).

---

## 2. Layer-by-Layer Breakdown

### A. Encoder Block (`enc1`, `enc2`, `enc3`)
* **`nn.Conv2d(..., stride=2, padding=1)`:** Halves the spatial dimensions ($H \times W$) while increasing channel capacity. Downsampling via strided convolutions is preferred over MaxPooling because the network learns its own spatial reduction filter weights.
* **`nn.BatchNorm2d`:** Normalizes activations per channel across the mini-batch, stabilizing gradients and allowing faster convergence.
* **`nn.ReLU(inplace=True)`:** Introduces non-linearity; `inplace=True` reduces memory allocation overhead by modifying the tensor in-place.

### B. Bottleneck (`bottleneck`)
* Reaches the smallest spatial dimension ($2 \times 2$) with $128$ feature channels.
* Forces the model to extract the most condensed, high-level feature representations of the image while discarding global background noise.

### C. Decoder & Skip Connections (`up1`-`up3`, `dec1`-`dec3`)
The decoding process operates in three synchronized steps for each resolution tier:

1. **Upsampling (`upX` - `nn.ConvTranspose2d`):** 
   * Doubles the spatial resolution ($2\times2 \to 4\times4 \to 8\times8 \to 16\times16$) using transposed convolutions (`stride=2, padding=1, output_padding=1`).
2. **Feature Concatenation (`torch.cat(..., dim=1)`):**
   * Concatenates the upsampled features with the corresponding encoder layer along the channel dimension (`dim=1`).
   * *Example at Level 1:* `d1` (`64` channels) + `e3` (`64` channels) $\to$ `128` total channels.
3. **Feature Refinement (`decX` - `nn.Conv2d`):**
   * Passes the concatenated $128$-channel tensor through a standard $3 \times 3$ convolution (`stride=1, padding=1`) to compress channels back down ($128 \to 64$) and fuse skip-connection details with upsampled features.

### D. Final Reconstruction Layer (`final_up` & `final_act`)
* **`final_up`:** Upsamples from $[16, 16, 16]$ back to the original image dimensions $[3, 32, 32]$.
* **`final_act` (`nn.Sigmoid`):** Maps output values to the normalized range $[0.0, 1.0]$, matching the expected range of normalized input images.

---

## 3. Key Takeaways

1. **Why channel dimensions double during `cat`:** The decoder layer (`decX`) must take `in_channels` equal to `out_channels_from_up + out_channels_from_enc`.
2. **Elimination of Blurring:** The skip connections bypass the extreme bottleneck, supplying exact high-frequency visual details directly to the reconstruction layers.

In [ ]:
class UNET_Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        # [3, 32, 32] -> [16, 16, 16]
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=(3, 3), stride=2, padding=1),
            nn.BatchNorm2d(num_features=16),
            nn.ReLU(inplace=True)
        )

        # [16, 16, 16] -> [32, 8, 8]
        self.enc2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3, 3), stride=2, padding=1),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(inplace=True)
        )

        # [32, 8, 8] -> [64, 4, 4]
        self.enc3 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3, 3), stride=2, padding=1),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(inplace=True)
        )

        # Bottleneck
        # [64, 4, 4] -> [128, 2, 2]
        self.bottleneck = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(3, 3), stride=2, padding=1),
            nn.BatchNorm2d(num_features=128),
            nn.ReLU()
        )

        # Decoder
        self.up1 = nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=(3, 3), stride=2, padding=1, output_padding=1)

        self.dec1 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=64, kernel_size=(3,3), stride=1, padding=1),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(inplace=True)
        )


        self.up2 = nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=(3, 3), stride=2, padding=1, output_padding=1)
        
        self.dec2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=32, kernel_size=(3, 3), stride=1, padding=1),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(inplace=True)
        )


        self.up3 = nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=(3, 3), stride=2, padding=1, output_padding=1)

        self.dec3 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=16, kernel_size=(3, 3), stride=1, padding=1),
            nn.BatchNorm2d(num_features=16),
            nn.ReLU(inplace=True)
        )

        self.final_up = nn.ConvTranspose2d(in_channels=16, out_channels=3, kernel_size=(3, 3), stride=2, padding=1, output_padding=1)
        self.final_act = nn.Sigmoid()

    def forward(self, x):
        e1 = self.enc1(x)       # [batch, 16, 16, 16]
        e2 = self.enc2(e1)      # [batch, 32, 8, 8]
        e3 = self.enc3(e2)      # [batch, 64, 4, 4]

        b = self.bottleneck(e3)     # [batch, 128, 2, 2]

        d1 = self.up1(b)
        d1 = torch.cat([d1, e3], dim=1)     # Connect on channels: 64+64=128
        d1 = self.dec1(d1)

        d2 = self.up2(d1)
        d2 = torch.cat([d2, e2], dim=1)     # Connect on channels: 32+32=64
        d2 = self.dec2(d2)

        d3 = self.up3(d2)
        d3 = torch.cat([d3, e1], dim=1)     # Connect on channels: 16+16=32
        d3 = self.dec3(d3)

        out = self.final_up(d3)
        out = self.final_act(out)

        return out



## Architectural Overview: Hybrid Loss Functions

This section breaks down the two custom hybrid loss functions used to improve the reconstruction quality and sharpness of the autoencoder. 

---

## 1. CombinedLoss (MSE + L1)

The `CombinedLoss` merges Mean Squared Error (MSE / $L_2$) and Mean Absolute Error (MAE / $L_1$) to balance overall pixel accuracy with edge preservation.

### High-Level Concept
* **MSE (`nn.MSELoss`):** Squares the errors, which heavily penalizes large deviations but ignores small differences. This leads to safe, averaged pixel values, causing the notorious **blurring effect** in autoencoders.
* **L1 (`nn.L1Loss`):** Penalizes errors linearly. It does not over-smooth the image, helping to preserve sharper edges and transitions.
* **The Hybrid Approach:** By combining them, MSE anchors the overall color and brightness, while L1 forces the network to maintain crisp boundaries.

### Implementation Details
* **`alpha` (Balance Weight):** Controls the contribution of each loss. For example, `alpha = 0.5` gives equal weight to both. Lowering `alpha` (e.g., `0.3`) forces the model to rely more on L1, reducing blur.
* **Calculation:** `total_loss = alpha * mse_loss + (1 - alpha) * l1_loss`

In [ ]:
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.mse = nn.MSELoss()
        self.l1 = nn.L1Loss()
        self.alpha = alpha      # balance weight

    def forward(self, prediction, targets):
        mse_loss = self.mse(prediction, targets)
        l1_loss = self.l1(prediction, targets)

        total_loss = self.alpha * mse_loss + (1-self.alpha) * l1_loss

        return total_loss

---

## 2. L1SSIMLoss (L1 + SSIM)

The `L1SSIMLoss` is an advanced hybrid function that combines structural human-like perception (SSIM) with absolute pixel accuracy (L1). It is considered a gold standard for image restoration and denoising.

### High-Level Concept
* **SSIM (Structural Similarity Index):** Instead of comparing isolated pixels, SSIM evaluates local windows of pixels to measure structural patterns, contrast, and luminance. It mimics how the human eye perceives image quality. 
* **Translating SSIM to Loss:** Since the SSIM metric returns `1.0` for identical images, the objective function is inverted as `1 - SSIM` (minimizing the loss drives SSIM towards 1).
* **The Hybrid Approach:** SSIM excels at preserving high-frequency textures and structures but can sometimes shift absolute color values. The L1 loss is added to anchor the exact color matching.

### Implementation Details
* **`SSIM(data_range=1.0, channel=3)`:** Configured for normalized RGB images where pixel values fall between `[0.0, 1.0]` across 3 channels.
* **`alpha = 0.84`:** A standard empirical weight often used in research (e.g., by Google) to heavily prioritize structural fidelity ($84\%$) while keeping a small L1 contribution ($16\%$) to prevent color drifting.
* **Calculation:** `total_loss = alpha * (1 - ssim) + (1 - alpha) * l1_loss`

In [ ]:
from pytorch_msssim import SSIM

class L1SSIMLoss(nn.Module):
    def __init__(self, alpha=0.84):
        super().__init__()
        self.ssim = SSIM(data_range=1.0, size_average=True, channel=3)
        self.l1 = nn.L1Loss()
        self.alpha = alpha

    def forward(self, predictions, targets):
        ssim_loss = 1 - self.ssim(predictions, targets)
        l1_loss = self.l1(predictions, targets)
        
        return self.alpha * ssim_loss + (1 - self.alpha) * l1_loss

In [ ]:
model = UNET_Autoencoder()
model.to(device=DEVICE)

criterion = L1SSIMLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

In [ ]:
model.train()
for epoch in range(EPOCHS):
    train_loss = 0.0
    for images, _ in train_dataloader:
        images = images.to(device=DEVICE)
        noisy_images = images + NOISE_FACTOR * torch.randn_like(images)
        noisy_images = torch.clamp(noisy_images, 0., 1.)

        outputs = model(noisy_images)
        loss = criterion(outputs, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch: {epoch+1}/{EPOCHS}, Loss: {train_loss}")

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    dataiter = iter(test_dataloader)
    images, _ = next(dataiter)

    images = images.to(device=DEVICE)

    noisy_images = images + NOISE_FACTOR * torch.randn_like(images)
    noisy_images = torch.clamp(noisy_images, 0., 1.)

    outputs = model(noisy_images)

    images = images.cpu().numpy()
    noisy_images = noisy_images.cpu().numpy()
    outputs = outputs.cpu().numpy()

    fig, axes = plt.subplots(nrows=3, ncols=10, figsize=(12, 6))

    for i in range(10):
        orig_img = images[i].transpose(1, 2, 0)
        noisy_img = noisy_images[i].transpose(1, 2, 0)
        denoised_img = outputs[i].transpose(1, 2, 0)

        axes[0, i].imshow(orig_img)
        axes[0, i].axis("off")
        axes[0, i].set_title("Original")

        axes[1, i].imshow(noisy_img)
        axes[1, i].axis("off")
        axes[1, i].set_title("Noisy")

        axes[2, i].imshow(denoised_img)
        axes[2, i].axis("off")
        axes[2, i].set_title("Denoised")

    plt.tight_layout()
    plt.show()
